In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_esquema = "movie_gold"
v_tabla = "result_group_movie_genre"
v_partition = "file_date"
v_merge_condition = "target.year_release_date = source.year_release_date and target.genre_name = source.genre_name"

In [0]:
#Extracion de datos
##Tabla de carga completa
genre_df = spark.read.table("movie_silver.genres")

#Tablas con data particionada
movies_df = spark.read.table("movie_silver.movies")\
                      .filter(
                               (col("file_date") == f"{v_file_date}")
                             )
##                                 
movie_genre_df = spark.read.table("movie_silver.movies_genres")\
                           .filter(
                               (col("file_date") == f"{v_file_date}")
                             )



In [0]:
#Tabla de movies filtrada con los campos y datos que necesitamos

movies_df = movies_df.select(movies_df.movie_id,
                            movies_df.year_release_date, 
                            movies_df.budget,
                            movies_df.revenue
                            )


In [0]:
#eneramos la tabla agregada con los campos solicitados
genre_movie_df = genre_df.join( movie_genre_df,
                                genre_df.genre_id == movie_genre_df.genre_id,
                                "inner"
                               )\
                          .select(movie_genre_df.movie_id, genre_df.genre_name)


movies_df = movies_df.filter(
                               (col("year_release_date") >= "2015")
                             )

movies_genre_final_df = movies_df.join(genre_movie_df,
                                       movies_df.movie_id == genre_movie_df.movie_id,
                                       "inner"
                                       )\
                                 .select(movies_df["*"], genre_movie_df.genre_name)


In [0]:

movies_genre_agg_df = movies_genre_final_df.groupBy("year_release_date", "genre_name")\
                                           .agg(
                                                 sum("budget").alias("budget"),
                                                 sum("revenue").alias("revenue")   
                                               )                                                 

In [0]:
#"movies_genre_agg_df.select("year_release_date")
result_group_movie_genre_df = movies_genre_agg_df.select( "year_release_date","genre_name", "budget", "revenue")\
                                                 .withColumn("dense_rank", dense_rank().over(Window.partitionBy("year_release_date")
                                                                                                   .orderBy(desc("budget"))
                                                                                                   .orderBy(desc("revenue"))
                                                                                             )
                                                     )
                                        

In [0]:

result_group_movie_genre_df = add_ingestion_date(result_group_movie_genre_df)
result_group_movie_genre_df = add_env(result_group_movie_genre_df)
final_df = add_file_date (result_group_movie_genre_df)

In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
resultado = merge_delta_lake (v_esquema, v_tabla, final_df, v_merge_condition, v_partition)
print(resultado)

In [0]:
#Guardamos en la capa gold 
#result_group_movie_genre_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
#print(f"Se insertaron {result_group_movie_genre_df.count()} registros en la tabla {v_esquema}.{v_tabla}")
